In [ ]:
# pip install requests-aws4auth

In [ ]:
import json
import pprint

import requests

In [ ]:
OPENSEARCH_HOST = "https://hogehoge:10004"
OPENSEARCH_USER = "OpenSearchのユーザID"
OPENSEARCH_PASSWORD = "OpenSearchのパスワード"

In [ ]:
REGION = "us-east-1"  # amazon nova multimodal embedding(リージョンはus-east-1のみ)
# REGION = "ap-northeast-1"  # claude 4.6 sonnet
SERVICE_NAME = "bedrock"
BEDROCK_ENDPOINT = "https://bedrock-runtime.us-east-1.amazonaws.com/model/amazon.nova-2-multimodal-embeddings-v1:0/invoke"  # amazon nova multimodal embedding(リージョンはus-east-1のみ)
# BEDROCK_ENDPOINT = "https://bedrock-runtime.ap-northeast-1.amazonaws.com/model/jp.anthropic.claude-sonnet-4-6/invoke"  # claude 4.6 sonnet # https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-sonnet-4-6.html
ACCESS_KEY_FOR_BEDROCK = "Amazon Bedrockに触れるIAMユーザのaccessキー"
SECRET_KEY_FOR_BEDROCK = "Amazon Bedrockに触れるIAMユーザのsecretキー"

In [ ]:
response = requests.get(url=OPENSEARCH_HOST,
                        auth=(OPENSEARCH_USER,
                              OPENSEARCH_PASSWORD),
                        headers={"Content-Type": "application/json"},
                        verify=False
                       )

In [ ]:
print(response.status_code)
pprint.pprint(response.json())

##### https://docs.opensearch.org/latest/vector-search/ai-search/agentic-search/neural-search/

### 登録する
##### https://docs.opensearch.org/latest/ml-commons-plugin/remote-models/index/

##### ML Connectorを作成
##### https://docs.opensearch.org/latest/ml-commons-plugin/remote-models/index/#step-2-create-a-connector
##### https://docs.opensearch.org/latest/ml-commons-plugin/remote-models/connectors/#amazon-bedrock-connector
##### https://dev.classmethod.jp/articles/20260621-amazon-opensearch-serverless-agentic-search/
##### https://docs.opensearch.org/latest/vector-search/ai-search/agentic-search/agent-customization/#claude-4-sonnet
##### https://docs.opensearch.org/latest/ml-commons-plugin/remote-models/supported-connectors/

In [ ]:
# amazon nova multimodal embedding
# request_body部分の詳細は https://docs.aws.amazon.com/nova/latest/userguide/embeddings-schema.html
# ML Connectorのpayloadのblueprintは https://github.com/opensearch-project/ml-commons/blob/main/docs/remote_inference_blueprints/bedrock_connector_nova_multimodal_model_blueprint.md
connector_payload = {"name": "nova-multimodal-embeddings-in-us-east-1",
                     "description": "Amazon Nova Multimodal Embeddings",
                     "version": 1,
                     "protocol": "aws_sigv4",
                     "credential": {"access_key": ACCESS_KEY_FOR_BEDROCK,
                                    "secret_key": SECRET_KEY_FOR_BEDROCK,},
                     "parameters": {"region": REGION,
                                    "service_name": SERVICE_NAME,
                                    "model": "amazon.nova-2-multimodal-embeddings-v1:0",
                                    "input_docs_processed_step_size": 1,
                                    "dimensions": 1024,
                                    "normalize": True,
                                    "embeddingTypes": ["float"],
                                    "truncationMode": "END",
                                    "imageFormat": "jpeg",
                                    "detailLevel": "DOCUMENT_IMAGE",
                                    "videoFormat": "mp4",
                                    "embeddingMode": "AUDIO_VIDEO_COMBINED",
                                    "audioFormat": "mp3"},
                     "actions": [{"action_type": "predict",
                                  "method": "POST",
                                  "headers": {"content-type": "application/json",
                                              "x-amz-content-sha256": "required"},
                                  "url": "https://bedrock-runtime.${parameters.region}.amazonaws.com/model/${parameters.model}/invoke",
                                  "request_body": "{\n  \"taskType\": \"SINGLE_EMBEDDING\",\n  \"singleEmbeddingParams\": {\n    \"embeddingPurpose\": \"GENERIC_INDEX\",\n    \"embeddingDimension\": ${parameters.dimensions},\n    \"text\": {\n      \"truncationMode\": \"${parameters.truncationMode:-null}\",\n      \"value\": \"${parameters.text:-null}\"\n    },\n    \"image\": {\n      \"detailLevel\": \"${parameters.detailLevel:-null}\",\n      \"format\": \"${parameters.imageFormat:-null}\",\n      \"source\": {\n        \"bytes\": \"${parameters.image:-null}\"\n      }\n    },\n    \"video\": {\n      \"format\": \"${parameters.videoFormat:-null}\",\n      \"source\": {\n        \"bytes\": \"${parameters.video:-null}\"\n      },\n      \"embeddingMode\": \"${parameters.embeddingMode:-null}\"\n    },\n    \"audio\": {\n      \"format\": \"${parameters.audioFormat:-null}\",\n      \"source\": {\n        \"bytes\": \"${parameters.audio:-null}\"\n      }\n    }\n  }\n}",
                                  "pre_process_function": "connector.pre_process.bedrock.nova.embedding",
                                  "post_process_function": "connector.post_process.bedrock.nova.embedding"}]
                    }

In [ ]:
# claude 4.6 sonnet
# request_body部分の詳細はhttps://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-anthropic-claude-messages-request-response.html
# ML Connectorのpayloadのblueprintは https://github.com/opensearch-project/ml-commons/blob/main/docs/remote_inference_blueprints/bedrock_connector_anthropic_claude4_blueprint.md
# connector_payload = {"name": "query-planner-claude-4.6-sonnet-in-ap-northeast-1",
#                      "description": "Query Planner Claude 4.6 Sonnet ",
#                      "version": 1,
#                      "protocol": "aws_sigv4",
#                      "credential": {"access_key": ACCESS_KEY_FOR_BEDROCK,
#                                     "secret_key": SECRET_KEY_FOR_BEDROCK},
#                      "parameters": {"region": REGION,
#                                     "service_name": SERVICE_NAME,
#                                     "model": "jp.anthropic.claude-sonnet-4-6"},
#                      "actions": [{"action_type": "predict",
#                                   "method": "POST",
#                                   "url": BEDROCK_ENDPOINT,
#                                   "headers": {"content-type": "application/json"},
#                                   "request_body": "{ \"system\": [{\"text\": \"${parameters.system_prompt}\"}], \"messages\": [${parameters._chat_history:-}{\"role\":\"user\",\"content\":[{\"text\":\"${parameters.user_prompt}\"}]}${parameters._interactions:-}]${parameters.tool_configs:-} }"
#                                  }]
#                     }

In [ ]:
connector_register_url = "{a}/_plugins/_ml/connectors/_create".format(a=OPENSEARCH_HOST)
response = requests.post(connector_register_url,
                         auth=(OPENSEARCH_USER,
                               OPENSEARCH_PASSWORD),
                         headers={"Content-Type": "application/json"},
                         json=connector_payload,
                         verify=False
                        )
print(response.status_code)
print(type(response.text))
print(response.text)

In [ ]:
connector_id = json.loads(response.text)["connector_id"]

##### ML Modelを作成
##### https://docs.opensearch.org/latest/ml-commons-plugin/remote-models/index/#step-3-register-an-externally-hosted-model

In [ ]:
# amazon nova multimodal embedding
ml_model_payload = {"name": "amazon-nova-multimodal-embedding-v1-in-us-east-1",
                    "version": "1.0.0",
                    "description": "Amazon Nova Multimodal Embeddings model via Bedrock connector",
                    "function_name": "remote",
                    "connector_id": connector_id
                   }
# claude 4.6 sonnet
# ml_model_payload = {"name": "query-planner-claude-4.6-sonnet-in-ap-northeast-1",
#                     "version": "1.0.0",
#                     "description": "Query Planner Claude 4.6 Sonnet via Bedrock connector",
#                     "function_name": "remote",
#                     "connector_id": connector_id
#                    }

In [ ]:
ml_model_register_url = "{a}/_plugins/_ml/models/_register".format(a=OPENSEARCH_HOST)
response_2 = requests.post(ml_model_register_url,
                           auth=(OPENSEARCH_USER,
                                 OPENSEARCH_PASSWORD),
                           headers={"Content-Type": "application/json"},
                           json=ml_model_payload,
                           verify=False
                          )
print(response_2.status_code)
print(type(response_2.text))
print(response_2.text)

In [ ]:
task_id = json.loads(response_2.text)["task_id"]
model_id = json.loads(response_2.text)["model_id"]

In [ ]:
task_check_url = "{a}/_plugins/_ml/tasks/{b}".format(a=OPENSEARCH_HOST, b=task_id)
response_3 = requests.get(task_check_url,
                          auth=(OPENSEARCH_USER,
                                OPENSEARCH_PASSWORD),
                          verify=False
                         )
print(response_3.status_code)
print(response_3.text)

##### 作成したML Modelをデプロイ
##### https://docs.opensearch.org/latest/ml-commons-plugin/remote-models/index/#step-4-deploy-the-model

In [ ]:
ml_model_deploy_url = "{a}/_plugins/_ml/models/{b}/_deploy".format(a=OPENSEARCH_HOST, b=model_id)
response_4 = requests.post(ml_model_deploy_url,
                           auth=(OPENSEARCH_USER,
                                 OPENSEARCH_PASSWORD),
                           verify=False
                          )
print(response_4.status_code)
print(response_4.text)

In [ ]:
deploy_task_id = json.loads(response_4.text)["task_id"]

In [ ]:
ml_model_deploy_task_check_url = "{a}/_plugins/_ml/tasks/{b}".format(a=OPENSEARCH_HOST, b=deploy_task_id)
response_5 = requests.get(ml_model_deploy_task_check_url,
                          auth=(OPENSEARCH_USER,
                                OPENSEARCH_PASSWORD),
                          verify=False
                         )
print(response_5.status_code)
print(response_5.text)

### ML Modelをテストする
##### https://docs.opensearch.org/latest/ml-commons-plugin/remote-models/index/#step-5-optional-test-the-model
##### https://docs.opensearch.org/latest/ml-commons-plugin/api/train-predict/predict/

In [ ]:
test_ml_model_id = "テストするML ModelのID"  # amazon nova multimodal embedding
# test_ml_model_id = "テストするML ModelのID"  # claude 4.6 sonnet

In [ ]:
ml_model_test_url = "{a}/_plugins/_ml/models/{b}/_predict".format(a=OPENSEARCH_HOST, b=test_ml_model_id)

In [ ]:
# ここのpayloadは「connector作成時のスキーマで変数として引き込む形にしたもの」を変数として設定する
# 例えば、connectorのスキーマで"value":"${parameters.text}"と設定していたら、このpayloadでは{"parameters": {"text": 引き込ませる値}}となる
test_ml_model_payload = {"parameters": {"text": "OpenSearchを試す"}}  # amazon nova multimodal embedding
# test_ml_model_payload = {"parameters":{"system_prompt":"You are a helpful assistant. Answer concisely.",
#                                        "user_prompt":"Reply with exactly one word: hello"}}  # claude 4.6 sonnet

In [ ]:
response_6 = requests.post(ml_model_test_url,
                           auth=(OPENSEARCH_USER,
                                 OPENSEARCH_PASSWORD),
                           headers={"Content-Type": "application/json"},
                           json=test_ml_model_payload,
                           verify=False
                          )
print(response_6.status_code)
print(type(response_6.text))
# print(json.loads(response_6.text)["inference_results"][0]["output"][0]["dataAsMap"]["embeddings"][0]["embedding"])  # amazon nova multimodal embedding
print(response_6.text)

### 削除する

##### 対象のML Modelのデプロイを解除する

In [ ]:
delete_ml_model_id = "削除するML ModelのID"

In [ ]:
ml_model_undeploy_url = "{a}/_plugins/_ml/models/{b}/_undeploy".format(a=OPENSEARCH_HOST, b=delete_ml_model_id)
response_7 = requests.post(ml_model_undeploy_url,
                           auth=(OPENSEARCH_USER,
                                 OPENSEARCH_PASSWORD),
                           verify=False
                          )
print(response_7.status_code)
print(response_7.text)

##### 対象のML Modelを削除する

In [ ]:
ml_model_delete_url = "{a}/_plugins/_ml/models/{b}".format(a=OPENSEARCH_HOST, b=delete_ml_model_id)
response_8 = requests.delete(ml_model_delete_url,
                             auth=(OPENSEARCH_USER,
                                   OPENSEARCH_PASSWORD),
                             verify=False
                            )
print(response_8.status_code)
print(response_8.text)

##### 対象のML Connectorを削除する

In [ ]:
delete_connector_id = "削除するML ConnectorのID"

In [ ]:
connector_delete_url = "{a}/_plugins/_ml/connectors/{b}".format(a=OPENSEARCH_HOST, b=delete_connector_id)
response_9 = requests.delete(connector_delete_url,
                             auth=(OPENSEARCH_USER,
                                   OPENSEARCH_PASSWORD),
                             verify=False
                            )
print(response_9.status_code)
print(response_9.text)